# Evaluation Loop

Runs every active model on every active feature track, target, horizon and pseudo-out-of-sample fold, then saves per-run metrics, predictions and runtimes to `results/raw/<model>/<track>.parquet`.

This notebook only holds configuration. The logic lives in `src/`:

- `src/evaluation/` — folds, track preparation, metrics, the loop itself
- `src/models/` — model functions and `MODEL_REGISTRY`
- `src/features/quantum/` — quantum / classical reservoir feature generators (precomputed into tracks by `scripts/build_quantum_tracks.py`)

To add a model, see the docstring in `src/models/__init__.py`. To add a track, add an entry to `TRACK_CONFIG` below.

## 1. Environment setup

In [1]:
import os
import sys
from pathlib import Path

COLAB = False  # set to True when running in Google Colab

if COLAB:
    REPO_URL  = "https://github.com/manuelarceaguirre/capstone-quantum.git"
    BRANCH    = "clean"
    REPO_ROOT = Path("/content/capstone-quantum")

    %cd /content
    !rm -rf capstone-quantum
    !git clone -b {BRANCH} {REPO_URL}
    %cd /content/capstone-quantum
    !pip install -q pmdarima
    sys.path.insert(0, str(REPO_ROOT))
else:
    # local: walk up from the working directory to the repo root;
    # src/ is importable via `pip install -e .` (see environment.yml)
    REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())

print("Repo root:", REPO_ROOT)

Repo root: C:\dev\projects\capstone-quantum


In [2]:
import warnings

import pandas as pd

from src.evaluation import (
    disaggregate_by_usrec,
    explode_predictions,
    generate_folds,
    load_results,
    load_tracks,
    run_evaluation,
    shared_track,
    summarize,
    write_csv,
)
from src.models import MODEL_REGISTRY

warnings.filterwarnings("ignore")

## 2. Configuration

In [3]:
# --- Paths ---
DATA_DIR    = REPO_ROOT / "data" / "processed"
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# --- Window parameters ---
WINDOW_MONTHS = 120   # training window length
TEST_MONTHS   = 1     # test window length
STEP_MONTHS   = 1     # fold step size
MAX_FOLDS     = None  # set to a small int for a quick smoke test
RUN_BY        = None  # your name, recorded on every result row (default: OS user)
RESUME        = False # True: skip runs already saved in results/ (e.g. after a crash or
                      # Colab disconnect). Keep False after changing a model's code.

# --- What to run ---
HORIZON_LIST  = [1, 3, 6]
TARGET_LIST   = ["INDPRO", "PAYEMS", "CPIAUCSL", "S&P 500"]
QUANTUM_TRACKS = ["Q_su_pce", "Q_su_pauli", "Q_su_bitstring", "Q_su_single_bit", "Q_heisenberg_pce"]
ACTIVE_TRACKS = ["B", *QUANTUM_TRACKS]                         # any keys of TRACK_CONFIG
ACTIVE_MODELS = ["random_walk", "AR6", "linear_readout"]       # any keys of MODEL_REGISTRY

# --- Track registry: TRACK_CONFIG[track][target] = {"type", "path"} ---
TRACK_CONFIG = {
    "A":      shared_track("dataframe", DATA_DIR / "track_A_full.parquet", TARGET_LIST),
    "B":      {t: {"type": "dataframe", "path": DATA_DIR / f"track_b_{t}.parquet"} for t in TARGET_LIST},
    "C":      shared_track("pipeline", DATA_DIR / "track_C_pipe.pkl", TARGET_LIST),       # LASSO, tabled
    "D":      shared_track("pipeline", DATA_DIR / "track_D_pipe.pkl", TARGET_LIST),       # PCA ~30 factors
    "D_mini": shared_track("pipeline", DATA_DIR / "track_D_mini_pipe.pkl", TARGET_LIST),  # PCA 15 factors
    # reservoir tracks: built by scripts/build_quantum_tracks.py
    "QRC":    {t: {"type": "dataframe", "path": DATA_DIR / f"track_QRC_{t}.parquet"} for t in TARGET_LIST},
    "CRC":    {t: {"type": "dataframe", "path": DATA_DIR / f"track_CRC_{t}.parquet"} for t in TARGET_LIST},
    # quantum feature tracks: built by notebooks/quantum_data_prep.ipynb
    **{q: {t: {"type": "dataframe", "path": DATA_DIR / f"track_{q}_{t}.parquet"} for t in TARGET_LIST}
       for q in QUANTUM_TRACKS},
}

MODELS = {name: MODEL_REGISTRY[name] for name in ACTIVE_MODELS}

print(f"Window:   {WINDOW_MONTHS}m train / {TEST_MONTHS}m test / {STEP_MONTHS}m step")
print(f"Targets:  {TARGET_LIST}")
print(f"Horizons: {HORIZON_LIST}")
print(f"Tracks:   {ACTIVE_TRACKS}")
print(f"Models:   {ACTIVE_MODELS}")

Window:   120m train / 1m test / 1m step
Targets:  ['INDPRO', 'PAYEMS', 'CPIAUCSL', 'S&P 500']
Horizons: [1, 3, 6]
Tracks:   ['B', 'Q_su_pce', 'Q_su_pauli', 'Q_su_bitstring', 'Q_su_single_bit', 'Q_heisenberg_pce']
Models:   ['random_walk', 'AR6', 'linear_readout']


## 3. Load data and build folds

In [4]:
track_data       = load_tracks(TRACK_CONFIG, ACTIVE_TRACKS, TARGET_LIST)
stationary_panel = pd.read_parquet(DATA_DIR / "stationary_panel.parquet")
targets          = pd.read_parquet(DATA_DIR / "targets.parquet")
usrec            = pd.read_parquet(DATA_DIR / "usrec.parquet")

print(f"Stationary panel {stationary_panel.shape}  {stationary_panel.index.min().date()} -> {stationary_panel.index.max().date()}")
print(f"Targets          {targets.shape}")

  Track B        target INDPRO     loaded   shape=(767, 10)
  Track B        target PAYEMS     loaded   shape=(767, 10)
  Track B        target CPIAUCSL   loaded   shape=(767, 12)
  Track B        target S&P 500    loaded   shape=(767, 12)
  Track Q_su_pce target INDPRO     loaded   shape=(767, 9)
  Track Q_su_pce target PAYEMS     loaded   shape=(767, 9)
  Track Q_su_pce target CPIAUCSL   loaded   shape=(767, 9)
  Track Q_su_pce target S&P 500    loaded   shape=(767, 9)
  Track Q_su_pauli target INDPRO     loaded   shape=(767, 15)
  Track Q_su_pauli target PAYEMS     loaded   shape=(767, 15)
  Track Q_su_pauli target CPIAUCSL   loaded   shape=(767, 18)
  Track Q_su_pauli target S&P 500    loaded   shape=(767, 18)
  Track Q_su_bitstring target INDPRO     loaded   shape=(767, 32)
  Track Q_su_bitstring target PAYEMS     loaded   shape=(767, 32)
  Track Q_su_bitstring target CPIAUCSL   loaded   shape=(767, 64)
  Track Q_su_bitstring target S&P 500    loaded   shape=(767, 64)
  Track Q_su

In [5]:
# all feature panels share the stationary panel's monthly index
folds = generate_folds(stationary_panel.index, WINDOW_MONTHS, TEST_MONTHS, STEP_MONTHS)
if MAX_FOLDS is not None:
    folds = folds[:MAX_FOLDS]

print(f"Total folds: {len(folds)}")
print(f"  first: train {folds[0]['train_start'].date()} -> {folds[0]['train_end'].date()}, test {folds[0]['test_start'].date()}")
print(f"  last:  train {folds[-1]['train_start'].date()} -> {folds[-1]['train_end'].date()}, test {folds[-1]['test_start'].date()}")

Total folds: 647
  first: train 1962-04-01 -> 1972-03-01, test 1972-04-01
  last:  train 2016-02-01 -> 2026-01-01, test 2026-02-01


## 4. Run

Each track's results are saved to `results/raw/<model>/<track>.parquet` every 25 folds and when that track finishes. After a crash or disconnect, set `RESUME = True` and rerun: runs already saved are skipped.

In [6]:
results_df = run_evaluation(
    folds            = folds,
    tracks           = ACTIVE_TRACKS,
    targets          = TARGET_LIST,
    horizons         = HORIZON_LIST,
    models           = MODELS,
    targets_df       = targets,
    track_config     = TRACK_CONFIG,
    track_data       = track_data,
    stationary_panel = stationary_panel,
    run_by           = RUN_BY,
    results_dir      = RESULTS_DIR,  # checkpoint every 25 folds and at the end of each track
    resume           = RESUME,
)

n_err = results_df["error"].notna().sum()
print(f"Runs with errors: {n_err}")
if n_err:
    print(results_df.loc[results_df["error"].notna(), ["model", "track", "target", "error"]].drop_duplicates("error").head())

  Saved: raw/random_walk/B.parquet   new=300  total=7724
  Saved: raw/AR6/B.parquet   new=300  total=7724
  Saved: raw/linear_readout/B.parquet   new=300  total=7724


  1000/139752 runs complete · 6s elapsed · ~14m47s left


  Saved: raw/random_walk/B.parquet   new=300  total=7724
  Saved: raw/AR6/B.parquet   new=300  total=7724
  Saved: raw/linear_readout/B.parquet   new=300  total=7724


  2000/139752 runs complete · 14s elapsed · ~15m40s left


  Saved: raw/random_walk/B.parquet   new=300  total=7724
  Saved: raw/AR6/B.parquet   new=300  total=7724
  Saved: raw/linear_readout/B.parquet   new=300  total=7724


...

  138000/139752 runs complete · 16m15s elapsed · ~12s left


  Saved: raw/random_walk/Q_heisenberg_pce.parquet   new=300  total=7724
  Saved: raw/AR6/Q_heisenberg_pce.parquet   new=300  total=7724
  Saved: raw/linear_readout/Q_heisenberg_pce.parquet   new=300  total=7724


  139000/139752 runs complete · 16m22s elapsed · ~5s left


  Saved: raw/random_walk/Q_heisenberg_pce.parquet   new=224  total=7724
  Saved: raw/AR6/Q_heisenberg_pce.parquet   new=224  total=7724
  Saved: raw/linear_readout/Q_heisenberg_pce.parquet   new=224  total=7724
Loop complete. Total rows: 139032 in 16m23s


Runs with errors: 0


## 5. Summarize

Saving merges rather than overwrites: only the rows a run recomputed are replaced, so teammates running other models or tracks are unaffected.

In [7]:
# summaries cover everything on disk for the active models/tracks, not just this run
all_results = load_results(RESULTS_DIR, models=ACTIVE_MODELS, tracks=ACTIVE_TRACKS)
summary = summarize(all_results)
summary

mse       mae  median_ae  \
model       track           target   horizon                                  
AR6         B               CPIAUCSL 1        0.000006  0.001775   0.001775   
                                     3        0.000042  0.004457   0.004457   
                                     6        0.000125  0.007908   0.007908   
                            INDPRO   1        0.000201  0.006012   0.006012   
                                     3        0.000676  0.012879   0.012879   
...                                                ...       ...        ...   
random_walk Q_su_single_bit PAYEMS   3        0.000253  0.003916   0.003916   
                                     6        0.000411  0.007583   0.007583   
                            S&P 500  1        0.002099  0.033552   0.033552   
                                     3        0.009329  0.071455   0.071455   
                                     6        0.021378  0.105361   0.105361   

                                                  rmse      mase     rmspe  
model       track           target   horizon                                
AR6         B               CPIAUCSL 1        0.001775  0.938839  1.347292  
                                     3        0.004457  0.950166  2.048899  
                                     6        0.007908  1.129897  1.292412  
                            INDPRO   1        0.006012  0.941258  4.458508  
                                     3        0.012879  1.062768  3.371093  
...                                                ...       ...       ...  
random_walk Q_su_single_bit PAYEMS   3        0.003916  1.336593  1.115775  
                                     6        0.007583  1.326675  2.578746  
                            S&P 500  1        0.033552  1.021314  6.237010  
                                     3        0.071455  1.031492  3.236078  
                                     6        0.105361  1.031899  3.795677  

[216 rows x 6 columns]

In [8]:
# recession (1) vs expansion (0)
summary_disagg = disaggregate_by_usrec(load_results(RESULTS_DIR), usrec)  # all models/tracks on disk
write_csv(summary_disagg, RESULTS_DIR / "summary_disaggregated.csv")
summary_disagg

mse       mae  \
model       track           target   horizon usrec                       
AR6         B               CPIAUCSL 1       0      0.000005  0.001586   
                                             1      0.000019  0.003242   
                                     3       0      0.000027  0.003785   
                                             1      0.000161  0.009629   
                                     6       0      0.000093  0.006961   
...                                                      ...       ...   
random_walk Q_su_single_bit S&P 500  1       1      0.006040  0.060672   
                                     3       0      0.007262  0.063212   
                                             1      0.025256  0.134947   
                                     6       0      0.014246  0.091439   
                                             1      0.076023  0.212038   

                                                    median_ae      rmse  \
model       track           target   horizon usrec                        
AR6         B               CPIAUCSL 1       0       0.001586  0.001586   
                                             1       0.003242  0.003242   
                                     3       0       0.003785  0.003785   
                                             1       0.009629  0.009629   
                                     6       0       0.006961  0.006961   
...                                                       ...       ...   
random_walk Q_su_single_bit S&P 500  1       1       0.060672  0.060672   
                                     3       0       0.063212  0.063212   
                                             1       0.134947  0.134947   
                                     6       0       0.091439  0.091439   
                                             1       0.212038  0.212038   

                                                        mase     rmspe  
model       track           target   horizon usrec                      
AR6         B               CPIAUCSL 1       0      0.860288  1.348996  
                                             1      1.546022  1.333686  
                                     3       0      0.823602  2.077020  
                                             1      1.925051  1.826281  
                                     6       0      1.001412  1.211426  
...                                                      ...       ...  
random_walk Q_su_single_bit S&P 500  1       1      1.757076  3.168137  
                                     3       0      0.918361  3.346871  
                                             1      1.902908  2.382672  
                                     6       0      0.906439  4.047363  
                                             1      1.993192  1.874021  

[456 rows x 6 columns]

In [9]:
# one row per (run, test date): y_true vs y_pred
predictions = explode_predictions(all_results)
runtime = all_results.groupby(["model", "track"])["runtime_sec"].agg(["mean", "sum"]).round(3)
display(predictions.head(), runtime)

,model,track,target,horizon,fold_id,date,y_true,y_pred
0,AR6,B,CPIAUCSL,1,0,1972-04-01,0.002407,0.002366
1,AR6,B,CPIAUCSL,1,1,1972-05-01,0.002401,0.002529
2,AR6,B,CPIAUCSL,1,2,1972-06-01,0.002395,0.002931
3,AR6,B,CPIAUCSL,1,3,1972-07-01,0.002389,0.002682
4,AR6,B,CPIAUCSL,1,4,1972-08-01,0.004762,0.001904


mean      sum
model          track                           
AR6            B                 0.002   15.451
               Q_heisenberg_pce  0.002   14.754
               Q_su_bitstring    0.002   13.526
               Q_su_pauli        0.002   14.780
               Q_su_pce          0.002   14.861
               Q_su_single_bit   0.002   14.747
linear_readout B                 0.012   96.214
               Q_heisenberg_pce  0.012   90.801
               Q_su_bitstring    0.013  103.343
               Q_su_pauli        0.012   94.061
               Q_su_pce          0.012   92.534
               Q_su_single_bit   0.012   90.240
random_walk    B                 0.000    1.124
               Q_heisenberg_pce  0.000    1.092
               Q_su_bitstring    0.000    1.037
               Q_su_pauli        0.000    1.114
               Q_su_pce          0.000    1.085
               Q_su_single_bit   0.000    1.074